# Анализ воронки конверсий

## Назначение

Описательный расчёт вложенной воронки и условных конверсий на уровне заявок до построения панели DiD: абсолютные объёмы, переходы между этапами, контроль качества временных метрик. Формулы условных конверсий:

- $P(\text{sch}=1) = \frac{1}{N}\sum_i \text{sch\_flg}_i$
- $P(\text{meet}=1 \mid \text{sch}=1) = \frac{\sum_i \text{meet\_flg}_i \cdot \text{sch\_flg}_i}{\sum_i \text{sch\_flg}_i}$
- $P(\text{success}=1 \mid \text{meet}=1) = \frac{\sum_i \text{success\_flg}_i \cdot \text{meet\_flg}_i}{\sum_i \text{meet\_flg}_i}$
- $P(\text{utlz}=1 \mid \text{success}=1) = \frac{\sum_i \text{utlz\_flg}_i \cdot \text{success\_flg}_i}{\sum_i \text{success\_flg}_i}$

## Входные данные

- `data/raw/application_dataset.csv`;
- флаги `sch_flg`, `meet_flg`, `success_flg`, `utlz_flg` и временные поля заявки.

## Результаты

- `outputs/final/conversion_funnel_summary.csv`, `conversion_funnel_steps.csv`;
- таблицы условных и безусловных долей, диагностика аномалий дат.

## Статус

**Канонический расчёт** — эталон описательной воронки; не заменяет каузальный DiD.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_FINAL = PROJECT_ROOT / "outputs" / "final"
OUT_FINAL.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных

In [ ]:
applications = pd.read_csv(
    "../data/raw/application_dataset.csv",
    sep=";",
    parse_dates=[
        "request_timestamp",
        "real_utilization_dttm",
        "planned_start_date",
        "first_success_dttm",
        "start_interval",
    ],
)

hexagons = pd.read_csv(
    "../data/raw/hexagons_dataset.csv",
    parse_dates=["treatment_date"],
)

print("Размер таблицы заявок:", applications.shape)
applications_schema = pd.DataFrame(
    {
        "столбец": applications.columns,
        "тип данных": applications.dtypes.astype(str).to_numpy(),
    }
)
display(applications_schema)

## 2. Обзор данных

In [3]:
flag_cols = ["sch_flg", "meet_flg", "success_flg", "utlz_flg"]
date_cols = [
    "request_timestamp",
    "real_utilization_dttm",
    "planned_start_date",
    "first_success_dttm",
    "start_interval",
]

data_overview = pd.Series(
    {
        "applications_rows": len(applications),
        "applications_unique_hex": applications["hex"].nunique(),
        "hexagons_rows": len(hexagons),
        "hexagons_unique_hex": hexagons["hex"].nunique(),
        "request_date_min": applications["request_timestamp"].min(),
        "request_date_max": applications["request_timestamp"].max(),
        "treatment_date_min": hexagons["treatment_date"].min(),
        "treatment_date_max": hexagons["treatment_date"].max(),
    }
).to_frame("value")

data_overview

,value
applications_rows,1807426
applications_unique_hex,35095
hexagons_rows,3644904
hexagons_unique_hex,3629072
request_date_min,2022-04-01 00:00:00
request_date_max,2022-10-19 00:00:00
treatment_date_min,2022-07-27 00:00:00
treatment_date_max,2022-10-19 00:00:00


## 3. Контроль качества данных

In [4]:
flag_quality = pd.DataFrame(
    {
        "missing": applications[flag_cols].isna().sum(),
        "not_0_or_1": applications[flag_cols].apply(
            lambda s: (~s.dropna().isin([0, 1])).sum()
        ),
        "share_1": applications[flag_cols].mean(),
    }
)

flag_quality

,missing,not_0_or_1,share_1
sch_flg,0,0,0.868521
meet_flg,0,0,0.676801
success_flg,0,0,0.657873
utlz_flg,0,0,0.494006


In [5]:
logic_checks = pd.Series(
    {
        "meet_without_scheduled": (
            (applications["meet_flg"] == 1) & (applications["sch_flg"] != 1)
        ).sum(),
        "success_without_meeting": (
            (applications["success_flg"] == 1) & (applications["meet_flg"] != 1)
        ).sum(),
        "utilization_without_success": (
            (applications["utlz_flg"] == 1) & (applications["success_flg"] != 1)
        ).sum(),
        "scheduled_without_planned_date": (
            (applications["sch_flg"] == 1) & (applications["planned_start_date"].isna())
        ).sum(),
        "planned_date_without_scheduled": (
            applications["planned_start_date"].notna() & (applications["sch_flg"] != 1)
        ).sum(),
        "success_without_success_date": (
            (applications["success_flg"] == 1) & (applications["first_success_dttm"].isna())
        ).sum(),
        "success_date_without_success": (
            applications["first_success_dttm"].notna() & (applications["success_flg"] != 1)
        ).sum(),
        "utilization_without_utilization_date": (
            applications["real_utilization_dttm"].isna() & (applications["utlz_flg"] == 1)
        ).sum(),
        "utilization_date_without_utilization": (
            (applications["utlz_flg"] != 1) & applications["real_utilization_dttm"].notna()
        ).sum(),
    }
).to_frame("number of anomalies")

logic_checks_display = logic_checks.rename(
    columns={"number of anomalies": "число аномалий"}
)
logic_checks_display

,число аномалий
meet_without_scheduled,0
success_without_meeting,413
utilization_without_success,56007
scheduled_without_planned_date,1287
planned_date_without_scheduled,18781
success_without_success_date,0
success_date_without_success,829
utilization_without_utilization_date,0
utilization_date_without_utilization,0


In [6]:
date_order_checks = pd.Series(
    {
        "planned_before_request": (
            applications["planned_start_date"] < applications["request_timestamp"]
        ).sum(),
        "first_slot_before_request": (
            applications["start_interval"] < applications["request_timestamp"]
        ).sum(),
        "success_before_request": (
            applications["first_success_dttm"] < applications["request_timestamp"]
        ).sum(),
        "utilization_before_request": (
            applications["real_utilization_dttm"] < applications["request_timestamp"]
        ).sum(),
    }
).to_frame("number of anomalies")

date_order_checks_display = date_order_checks.rename(
    columns={"number of anomalies": "число аномалий"}
)
date_order_checks_display

,число аномалий
planned_before_request,9801
first_slot_before_request,0
success_before_request,361
utilization_before_request,6088


## 4. Воронка конверсий

### 4.1 Абсолютные объёмы по этапам воронки

In [7]:
# Каноническая вложенная воронка. Каждый следующий этап является
# пересечением всех необходимых предыдущих событий; marginal utlz_flg
# сохраняется отдельно и не выдаётся за последний этап воронки.
flg = applications[flag_cols].fillna(0).astype("int8")

n_requests = len(applications)
n_scheduled = int(flg["sch_flg"].sum())
n_meetings = int((flg["sch_flg"] * flg["meet_flg"]).sum())
n_successful_meetings = int(
    (flg["sch_flg"] * flg["meet_flg"] * flg["success_flg"]).sum()
)
n_utilized_after_success = int(
    (
        flg["sch_flg"]
        * flg["meet_flg"]
        * flg["success_flg"]
        * flg["utlz_flg"]
    ).sum()
)
n_utilization_marginal = int(flg["utlz_flg"].sum())

funnel_steps = pd.DataFrame(
    {
        "step": [
            "requests",
            "scheduled",
            "meetings_after_scheduling",
            "successful_meetings",
            "utilized_after_success",
        ],
        "count": [
            n_requests,
            n_scheduled,
            n_meetings,
            n_successful_meetings,
            n_utilized_after_success,
        ],
    }
).set_index("step")

funnel_steps["lost_from_previous"] = (
    funnel_steps["count"].shift(1) - funnel_steps["count"]
)
funnel_steps["conv_from_previous"] = (
    funnel_steps["count"] / funnel_steps["count"].shift(1)
)
funnel_steps["conv_from_requests"] = funnel_steps["count"] / n_requests
funnel_steps["marginal_utlz_count"] = pd.NA
funnel_steps.loc["utilized_after_success", "marginal_utlz_count"] = (
    n_utilization_marginal
)

assert funnel_steps["count"].is_monotonic_decreasing
FUNNEL_STEP_LABELS = {
    "requests": "заявки",
    "scheduled": "назначенные встречи",
    "meetings_after_scheduling": "состоявшиеся встречи",
    "successful_meetings": "успешные встречи",
    "utilized_after_success": "утилизации после успеха",
}
FUNNEL_COLUMN_LABELS = {
    "count": "количество",
    "lost_from_previous": "потеря от предыдущего этапа",
    "conv_from_previous": "конверсия от предыдущего этапа",
    "conv_from_requests": "конверсия от заявок",
    "marginal_utlz_count": "маржинальное число утилизаций",
}
funnel_display = funnel_steps.rename(
    index=FUNNEL_STEP_LABELS,
    columns=FUNNEL_COLUMN_LABELS,
)
funnel_display

                количество  потеря от предыдущего этапа  конверсия от предыдущего этапа  \
step                                                            
заявки      1807426                 NaN                 NaN   
назначенные встречи     1569787            237639.0            0.868521   
состоявшиеся встречи      1223267            346520.0            0.779257   
успешные встречи     1188643             34624.0            0.971695   
утилизации   836873            351770.0            0.704057   

              конверсия от заявок  
step                              
заявки                1.000000  
назначенные встречи               0.868521  
состоявшиеся встречи                0.676801  
успешные встречи               0.657644  
утилизации            0.463019  

### 4.2 Условные конверсии (формулы §9.3.2)

In [8]:
# Условные конверсии используют те же вложенные числители, что и
# каноническая воронка. Маржинальная доля utlz_flg показана отдельно.
conditional_rates = pd.Series(
    {
        "P(sch=1)": n_scheduled / n_requests,
        "P(meet∩sch=1|sch=1)": (
            n_meetings / n_scheduled if n_scheduled > 0 else np.nan
        ),
        "P(success∩meet∩sch=1|meet∩sch=1)": (
            n_successful_meetings / n_meetings if n_meetings > 0 else np.nan
        ),
        "P(utlz∩success∩meet∩sch=1|success∩meet∩sch=1)": (
            n_utilized_after_success / n_successful_meetings
            if n_successful_meetings > 0
            else np.nan
        ),
        "P(utlz=1) [marginal]": n_utilization_marginal / n_requests,
    },
    name="conversion_rate",
).to_frame()

conditional_rates

,conversion_rate
P(sch=1),0.868521
P(meet=1|sch=1),0.779257
P(success=1|meet=1),0.971695
P(utlz=1|success=1),0.704057


### 4.3 Функция агрегированной воронки (с поддержкой группировки)

In [9]:
def summarize_funnel(data: pd.DataFrame, group_cols=None) -> pd.DataFrame:
    """Summarize a logically nested funnel and marginal utilization."""
    group_cols = [] if group_cols is None else list(group_cols)
    data = data.copy()
    for col in ["sch_flg", "meet_flg", "success_flg", "utlz_flg"]:
        data[col] = data[col].fillna(0).astype("int8")

    data["_meet_after_scheduling"] = data["sch_flg"] * data["meet_flg"]
    data["_successful_meeting"] = (
        data["_meet_after_scheduling"] * data["success_flg"]
    )
    data["_utilized_after_success"] = (
        data["_successful_meeting"] * data["utlz_flg"]
    )

    agg_cols = {
        "requests": ("sch_flg", "size"),
        "scheduled": ("sch_flg", "sum"),
        "meetings_after_scheduling": ("_meet_after_scheduling", "sum"),
        "successful_meetings": ("_successful_meeting", "sum"),
        "utilized_after_success": ("_utilized_after_success", "sum"),
        "utilization_marginal": ("utlz_flg", "sum"),
    }
    if group_cols:
        summary = data.groupby(
            group_cols, dropna=False, observed=False
        ).agg(**agg_cols)
    else:
        summary = pd.DataFrame(
            {
                "requests": [len(data)],
                "scheduled": [data["sch_flg"].sum()],
                "meetings_after_scheduling": [data["_meet_after_scheduling"].sum()],
                "successful_meetings": [data["_successful_meeting"].sum()],
                "utilized_after_success": [data["_utilized_after_success"].sum()],
                "utilization_marginal": [data["utlz_flg"].sum()],
            },
            index=["all"],
        )

    count_cols = list(agg_cols)
    summary[count_cols] = summary[count_cols].round().astype("int64")
    summary["scheduled_rate"] = summary["scheduled"] / summary["requests"]
    summary["meeting_rate_from_requests"] = (
        summary["meetings_after_scheduling"] / summary["requests"]
    )
    summary["success_rate_from_requests"] = (
        summary["successful_meetings"] / summary["requests"]
    )
    summary["utilization_after_success_rate_from_requests"] = (
        summary["utilized_after_success"] / summary["requests"]
    )
    summary["utilization_marginal_rate"] = (
        summary["utilization_marginal"] / summary["requests"]
    )
    summary["meeting_given_scheduled"] = (
        summary["meetings_after_scheduling"]
        / summary["scheduled"].replace(0, np.nan)
    )
    summary["success_given_meeting"] = (
        summary["successful_meetings"]
        / summary["meetings_after_scheduling"].replace(0, np.nan)
    )
    summary["utilization_given_success"] = (
        summary["utilized_after_success"]
        / summary["successful_meetings"].replace(0, np.nan)
    )
    assert (
        summary[
            [
                "requests",
                "scheduled",
                "meetings_after_scheduling",
                "successful_meetings",
                "utilized_after_success",
            ]
        ].diff(axis=1).iloc[:, 1:] <= 0
    ).all().all()
    return summary

In [10]:
funnel_summary = summarize_funnel(applications)
funnel_summary.reset_index(names="scope").to_csv(
    OUT_FINAL / "conversion_funnel_summary.csv", index=False
)
funnel_steps.reset_index().to_csv(
    OUT_FINAL / "conversion_funnel_steps.csv", index=False
)
funnel_summary

,requests,scheduled,meetings,successes,utilizations,scheduled_rate,meeting_rate_from_requests,success_rate_from_requests,utilization_rate_from_requests,meeting_given_scheduled,success_given_meeting,utilization_given_success
all,1807426,1569787,1223267,1188643,836873,0.868521,0.676801,0.657644,0.463019,0.779257,0.971695,0.704057


`first_slot_before_request = 0` — единственная чистая колонка. Основная временная метрика `T_available = start_interval - request_timestamp` не даёт отрицательных значений.

Основные аномалии, требующие решения до построения панели:
- `planned_before_request = 9801` — дата назначенной встречи раньше даты заявки; вспомогательная метрика $T_planned$ по этим строкам отрицательна и исключается из расчёта.
- `success_before_request = 361` — строки исключаются из расчёта `T_success`.
- `utilization_before_request = 6088` — вероятен временной сдвиг при гранулярности календарных дней; строки исключаются или обнуляются.

`funnel_steps` задаёт строго вложенную последовательность: заявки → назначения → встречи после назначения → успешные встречи → утилизации после успешной встречи. Поэтому `count` не возрастает по мере прохождения этапов, `conv_from_previous` является локальной условной конверсией, а `conv_from_requests` — накопленной конверсией.

Маржинальное число `utlz_flg=1` экспортируется отдельно как `marginal_utlz_count`. Оно не является последним этапом вложенной воронки: в данных возможен `utlz_flg=1` без наблюдаемого `success_flg=1`. Канонический последний этап — пересечение `sch_flg × meet_flg × success_flg × utlz_flg`.

`conditional_rates` разделяет четыре перехода вложенной воронки и маржинальную долю `utlz_flg`. Числитель каждого условного перехода включает все предыдущие события, поэтому логические аномалии флагов не разрушают монотонность. Маржинальная `P(utlz=1)` приводится только как отдельная описательная метрика и не интерпретируется как `P(utlz=1 | success=1)`.

`summarize_funnel`

Таблица объединяет всё вышесказанное в одну строку. Различие между колонками `_rate_from_requests` и `_given_`:

- `meeting_rate_from_requests = 0.677` - доля встреч от всех 1 807 426 заявок. Это безусловная вероятность, полезна для сравнения с другими периодами или гексагонами.
- `meeting_given_scheduled = 0.779` - доля встреч от тех, кому встреча уже была назначена. Это условная вероятность, диагностирует конкретный переход воронки.

Для операционных решений релевантны именно условные (`_given_`): они позволяют разделить ответственность между этапами - логистикой, клиентским поведением и операционным процессом.

## Вывод

На полной выборке (1.81 млн заявок) безусловная доля встреч от заявок — 67.7%, условный переход «встреча | назначение» — 77.9%, «успех | встреча» — 97.2%, «утилизация | успех» — 70.4%. Воронка монотонна по вложенным этапам; выявленные временные аномалии учтены при последующей сборке панели.